In [1]:
import json
import pandas as pd
import numpy as np
import re
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
primary_data = pd.read_csv(r"C:\Users\RICHARD\free_exercise_db.csv")

In [3]:
secondary_data = pd.read_csv(r"C:\Users\RICHARD\megaGymDataset.csv")

In [4]:
print(secondary_data.head())

   Unnamed: 0                         Title  \
0           0        Partner plank band row   
1           1  Banded crunch isometric hold   
2           2         FYR Banded Plank Jack   
3           3                 Banded crunch   
4           4                        Crunch   

                                                Desc      Type    BodyPart  \
0  The partner plank band row is an abdominal exe...  Strength  Abdominals   
1  The banded crunch isometric hold is an exercis...  Strength  Abdominals   
2  The banded plank jack is a variation on the pl...  Strength  Abdominals   
3  The banded crunch is an exercise targeting the...  Strength  Abdominals   
4  The crunch is a popular core exercise targetin...  Strength  Abdominals   

  Equipment         Level  Rating RatingDesc  
0     Bands  Intermediate     0.0        NaN  
1     Bands  Intermediate     NaN        NaN  
2     Bands  Intermediate     NaN        NaN  
3     Bands  Intermediate     NaN        NaN  
4     Bands  

In [9]:
print(primary_data.head())

                  id               name    category         level  equipment  \
0         3_4_Sit-Up         3/4 Sit-Up    strength      beginner  body only   
1    90_90_Hamstring    90/90 Hamstring  stretching      beginner  body only   
2  Ab_Crunch_Machine  Ab Crunch Machine    strength  intermediate    machine   
3          Ab_Roller          Ab Roller    strength  intermediate      other   
4           Adductor           Adductor  stretching  intermediate  foam roll   

    force   mechanic primary_muscles secondary_muscles  \
0    pull   compound      abdominals               NaN   
1    push        NaN      hamstrings            calves   
2    pull  isolation      abdominals               NaN   
3    pull   compound      abdominals         shoulders   
4  static  isolation       adductors               NaN   

                                        instructions  n_steps  \
0  1. Lie down on the floor and secure your feet....        5   
1  1. Lie on your back, with one leg ext

In [11]:
print(primary_data.shape)

(876, 13)


In [13]:
print(primary_data['category'].unique())

['strength' 'stretching' 'plyometrics' 'strongman' 'powerlifting' 'cardio'
 'olympic weightlifting']


In [15]:
print(secondary_data['Type'].unique())

['Strength' 'Plyometrics' 'Cardio' 'Stretching' 'Powerlifting' 'Strongman'
 'Olympic Weightlifting']


In [17]:
remove_category = ['strongman','powerlifting','olympic weightlifting']

primary_data = primary_data[~primary_data['category'].isin(remove_category)]

In [19]:
print(primary_data.shape)

(782, 13)


In [21]:
print(primary_data['level'].unique())

['beginner' 'intermediate' 'expert']


In [23]:
print(primary_data["equipment"].unique())

['body only' 'machine' 'other' 'foam roll' nan 'kettlebells' 'dumbbell'
 'cable' 'barbell' 'bands' 'medicine ball' 'exercise ball' 'e-z curl bar']


In [25]:
nan_equipment = primary_data[primary_data["equipment"].isna()]

display(nan_equipment)

,id,name,category,level,equipment,force,mechanic,primary_muscles,secondary_muscles,instructions,n_steps,image_1,image_2
5,Adductor_Groin,Adductor/Groin,stretching,intermediate,NaN,static,NaN,adductors,NaN,1. Lie on your back with your feet raised towa...,4,https://raw.githubusercontent.com/yuhonas/free...,https://raw.githubusercontent.com/yuhonas/free...
12,Alternate_Leg_Diagonal_Bound,Alternate Leg Diagonal Bound,plyometrics,beginner,NaN,push,compound,quadriceps,"abductors, adductors, calves, glutes, hamstrings",1. Assume a comfortable stance with one foot s...,4,https://raw.githubusercontent.com/yuhonas/free...,https://raw.githubusercontent.com/yuhonas/free...
20,Ankle_Circles,Ankle Circles,stretching,beginner,NaN,pull,isolation,calves,NaN,1. Use a sturdy object like a squat rack to ho...,3,https://raw.githubusercontent.com/yuhonas/free...,https://raw.githubusercontent.com/yuhonas/free...
21,Ankle_On_The_Knee,Ankle On The Knee,stretching,beginner,NaN,static,NaN,glutes,NaN,"1. From a lying position, bend your knees and ...",3,https://raw.githubusercontent.com/yuhonas/free...,https://raw.githubusercontent.com/yuhonas/free...
24,Arm_Circles,Arm Circles,stretching,beginner,NaN,push,isolation,shoulders,traps,1. Stand up and extend your arms straight out ...,3,https://raw.githubusercontent.com/yuhonas/free...,https://raw.githubusercontent.com/yuhonas/free...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
838,Upper_Back-Leg_Grab,Upper Back-Leg Grab,stretching,beginner,NaN,static,NaN,hamstrings,"lower back, middle back","1. While seated, bend forward to hug your thig...",2,https://raw.githubusercontent.com/yuhonas/free...,https://raw.githubusercontent.com/yuhonas/free...
839,Upper_Back_Stretch,Upper Back Stretch,stretching,beginner,NaN,static,NaN,middle back,middle back,1. Clasp fingers together with your thumbs poi...,1,https://raw.githubusercontent.com/yuhonas/free...,https://raw.githubusercontent.com/yuhonas/free...
843,Upward_Stretch,Upward Stretch,stretching,beginner,NaN,static,NaN,shoulders,"chest, lats","1. Extend both hands straight above your head,...",2,https://raw.githubusercontent.com/yuhonas/free...,https://raw.githubusercontent.com/yuhonas/free...
867,Windmills,Windmills,stretching,intermediate,NaN,pull,NaN,abductors,"glutes, hamstrings, lower back",1. Lie on your back with your arms extended ou...,3,https://raw.githubusercontent.com/yuhonas/free...,https://raw.githubusercontent.com/yuhonas/free...


In [27]:
primary_data["equipment"] = primary_data["equipment"].fillna("body only")

In [33]:
other_equipment = primary_data[
    primary_data["equipment"].astype(str).str.strip().str.lower() == "other"
]

display(other_equipment.head(50))

,id,name,category,level,equipment,force,mechanic,primary_muscles,secondary_muscles,instructions,n_steps,image_1,image_2
3,Ab_Roller,Ab Roller,strength,intermediate,other,pull,compound,abdominals,shoulders,1. Hold the Ab Roller with both hands and knee...,4,https://raw.githubusercontent.com/yuhonas/free...,https://raw.githubusercontent.com/yuhonas/free...
22,Anterior_Tibialis-SMR,Anterior Tibialis-SMR,stretching,intermediate,other,static,NaN,calves,NaN,1. Begin seated on the ground with your legs b...,2,https://raw.githubusercontent.com/yuhonas/free...,https://raw.githubusercontent.com/yuhonas/free...
33,Balance_Board,Balance Board,strength,beginner,other,NaN,compound,calves,"hamstrings, quadriceps",1. Place a balance board in front of you.\n2. ...,3,https://raw.githubusercontent.com/yuhonas/free...,https://raw.githubusercontent.com/yuhonas/free...
35,Band_Assisted_Pull-Up,Band Assisted Pull-Up,strength,beginner,other,NaN,compound,lats,"abdominals, forearms, middle back",1. Choke the band around the center of the pul...,4,https://raw.githubusercontent.com/yuhonas/free...,https://raw.githubusercontent.com/yuhonas/free...
67,Battling_Ropes,Battling Ropes,strength,beginner,other,push,compound,shoulders,"chest, forearms",1. For this exercise you will need a heavy rop...,4,https://raw.githubusercontent.com/yuhonas/free...,https://raw.githubusercontent.com/yuhonas/free...
69,Behind_Head_Chest_Stretch,Behind Head Chest Stretch,stretching,expert,other,static,isolation,chest,shoulders,1. Sit upright on the floor with your partner ...,4,https://raw.githubusercontent.com/yuhonas/free...,https://raw.githubusercontent.com/yuhonas/free...
75,Bench_Sprint,Bench Sprint,plyometrics,beginner,other,push,compound,quadriceps,"calves, glutes, hamstrings",1. Stand on the ground with one foot resting o...,4,https://raw.githubusercontent.com/yuhonas/free...,https://raw.githubusercontent.com/yuhonas/free...
87,Bicycling,Bicycling,cardio,beginner,other,NaN,NaN,quadriceps,"calves, glutes, hamstrings","1. To begin, seat yourself on the bike and adj...",1,https://raw.githubusercontent.com/yuhonas/free...,https://raw.githubusercontent.com/yuhonas/free...
93,Bodyweight_Mid_Row,Bodyweight Mid Row,strength,intermediate,other,pull,compound,middle back,"biceps, lats",1. Begin by taking a medium to wide grip on a ...,3,https://raw.githubusercontent.com/yuhonas/free...,https://raw.githubusercontent.com/yuhonas/free...
99,Box_Jump_Multiple_Response,Box Jump (Multiple Response),plyometrics,beginner,other,push,compound,hamstrings,"abductors, adductors, calves, glutes, quadriceps",1. Assume a relaxed stance facing the box or p...,3,https://raw.githubusercontent.com/yuhonas/free...,https://raw.githubusercontent.com/yuhonas/free...


In [35]:
primary_data = primary_data[
    primary_data["equipment"].astype(str).str.strip().str.lower() != "other"
]

In [37]:
non_ascii = primary_data["instructions"].fillna("").str.contains(r"[^\x00-\x7F]")
no_steps  = primary_data["instructions"].isna() | (primary_data["instructions"].str.strip() == "")
no_photos = primary_data["image_1"].isna() | primary_data["image_2"].isna()

print("Rows with unusual characters:", non_ascii.sum())
print("Rows with no instructions:   ", no_steps.sum())
print("Rows with no photos:         ", no_photos.sum())


Rows with unusual characters: 3
Rows with no instructions:    4
Rows with no photos:          3


In [39]:
REPLACEMENTS = {
    "¾": "3/4",
    "°": " degrees",
    "—": " - ",
    "–": " - ",
}

def clean_text(text):
    if pd.isna(text):
        return text
    for old, new in REPLACEMENTS.items():
        text = text.replace(old, new)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    return text.strip()

primary_data["instructions"] = primary_data["instructions"].apply(clean_text)

print("Rows with unusual characters after cleaning:",
      primary_data["instructions"].fillna("").str.contains(r"[^\x00-\x7F]").sum())

Rows with unusual characters after cleaning: 0


In [41]:
no_steps  = primary_data["instructions"].isna() | (primary_data["instructions"].str.strip() == "")
no_photos = primary_data["image_1"].isna() | primary_data["image_2"].isna()

removed = primary_data[no_steps | no_photos].copy()

primary_data = primary_data[~(no_steps | no_photos)].reset_index(drop=True)

print("Removed rows:", len(removed))

Removed rows: 7


## Step 8 - Remove duplicate names
Two rows are treated as duplicates when their names are the same after tidying
(lower case, no extra spaces). The first one is kept.

In [ ]:
def name_key(name):
    """Name used for comparing: lower case, single spaces, no surrounding spaces."""
    return re.sub(r"\s+", " ", str(name)).strip().lower()

before = len(primary_data)
primary_data["name_key"] = primary_data["name"].apply(name_key)
duplicates = primary_data[primary_data.duplicated("name_key", keep="first")]
primary_data = primary_data.drop_duplicates("name_key", keep="first").reset_index(drop=True)

print("Duplicate names removed:", before - len(primary_data))
display(duplicates[["name"]])
print("Rows now:", len(primary_data))

## Step 9 - Tidy text
Remove extra spaces from every text column, and write the category-type columns
(category, level, equipment, force, mechanic, muscles) in lower case so the same
value is always spelled the same way.

In [ ]:
text_columns = primary_data.select_dtypes(include="object").columns
for col in text_columns:
    if col in ("instructions", "image_1", "image_2"):
        continue
    primary_data[col] = primary_data[col].apply(
        lambda v: re.sub(r"\s+", " ", v).strip() if isinstance(v, str) else v)

lower_columns = ["category", "level", "equipment", "force", "mechanic",
                 "primary_muscles", "secondary_muscles"]
for col in lower_columns:
    primary_data[col] = primary_data[col].str.lower()

for col in ["category", "level", "equipment", "force", "mechanic"]:
    print(col, sorted(primary_data[col].dropna().unique()))

## Step 10 - Match the column format the app uses
The recommender expects the same columns and values it used before
(`exercise_category`, `difficulty_level`, `equipment_needed`, `target_muscle`,
`major_muscle_group`). The rules below are the same rules that were in the app's
`load_and_build_kb()`, applied to the new dataset:

* **Category**: cardio / plyometrics / mobility / flexibility / resistance (same keyword rule)
* **Level**: beginner / intermediate / advanced ("expert" becomes advanced)
* **Equipment**: "body only" becomes bodyweight, "dumbbell" becomes dumbbells, "bands" becomes resistance band
* **Muscle group**: the first primary muscle is grouped into chest, back, shoulders, quads,
  hamstrings, glutes, core, biceps, triceps, calves or other.
  (Fix: the old rule looked for "calf", which does not appear in "calves", so calf
  exercises were put in "other". It now checks for both.)

In [ ]:
def classify_category(raw_type, body, title):
    combined = f"{raw_type} {body} {title}".lower()
    if any(w in combined for w in ["cardio", "running", "cycling", "jump rope", "rowing machine"]):
        return "cardio"
    if "plyometric" in str(raw_type).lower():
        return "plyometrics"
    if any(w in combined for w in ["mobility", "dynamic", "warm-up", "warm up", "warmup"]):
        return "mobility"
    if any(w in combined for w in ["stretch", "flexibility", "yoga"]):
        return "flexibility"
    return "resistance"


def normalise_level(value):
    value = str(value).lower()
    if "beginner" in value or "novice" in value: return "beginner"
    if "intermediate" in value: return "intermediate"
    if "expert" in value or "advanced" in value: return "advanced"
    return "beginner"


EQUIPMENT_ALIASES = {"dumbbell": "dumbbells", "bands": "resistance band", "band": "resistance band"}

def normalise_equipment(value):
    value = str(value).strip().lower()
    if value in {"", "none", "nan", "body only", "bodyweight"}:
        return "bodyweight"
    return EQUIPMENT_ALIASES.get(value, value)


def major_muscle_group(value):
    text = str(value).lower()
    if any(t in text for t in ["chest", "pectoral"]): return "chest"
    if any(t in text for t in ["lat", "back", "rhomboid", "trap"]): return "back"
    if any(t in text for t in ["shoulder", "deltoid"]): return "shoulders"
    if any(t in text for t in ["quad", "leg"]): return "quads"
    if "hamstring" in text: return "hamstrings"
    if "glute" in text: return "glutes"
    if any(t in text for t in ["abdominal", "core", "oblique"]): return "core"
    if "bicep" in text: return "biceps"
    if "tricep" in text: return "triceps"
    if "calf" in text or "calves" in text: return "calves"     # fixed
    return "other"


first_muscle = primary_data["primary_muscles"].fillna("full body").str.split(",").str[0].str.strip()

primary_data["exercise_category"] = [classify_category(c, m, n) for c, m, n in
                                     zip(primary_data["category"], first_muscle, primary_data["name"])]
primary_data["difficulty_level"] = primary_data["level"].apply(normalise_level)
primary_data["equipment_needed"] = primary_data["equipment"].apply(normalise_equipment)
primary_data["target_muscle"] = first_muscle.str.title()
primary_data["major_muscle_group"] = first_muscle.apply(major_muscle_group)

display(pd.crosstab(primary_data["category"], primary_data["exercise_category"]))
print(primary_data["major_muscle_group"].value_counts())

## Step 11 - Assign where it can be done
Bodyweight exercises can be done at home, at the gym or outdoors.
Anything that needs equipment is gym only (same rule as before).

In [ ]:
ANYWHERE = ["home", "gym", "outdoor"]

def location_for(equipment):
    return ANYWHERE if equipment == "bodyweight" else ["gym"]

primary_data["location_suitability"] = primary_data["equipment_needed"].apply(location_for)
print(primary_data["location_suitability"].astype(str).value_counts())

## Step 12 - Label compound or isolation
Same keyword rule as the app: multi-joint lifts (squat, bench press, row ...) are compound,
single-joint movements (curl, raise, extension ...) are isolation. Only resistance
exercises can be compound. The dataset's own `mechanic` column is used as a check.

In [ ]:
def _contains_any(text, keywords):
    return any(re.search(r"\b" + re.escape(k), text) for k in keywords)

STRONG_COMPOUND = ["squat", "deadlift", "lunge", "bench press", "push-up", "pushup", "push up",
                   "pull-up", "pullup", "pull up", "chin-up", "chin up", "dip"]
ISOLATION = ["curl", "extension", "raise", "fly", "flye", "kickback", "crunch", "sit-up", "situp",
             "shrug", "pushdown", "push-down", "skull", "pullover", "plank", "wrist", "adduct",
             "abduct", "tricep", "bicep", "calf"]
WEAK_COMPOUND = ["press", "row", "pulldown", "pull-down", "thrust", "bridge", "clean", "snatch",
                 "step-up", "step up", "good morning", "swing", "burpee"]

def infer_compound(name, category):
    if category != "resistance":
        return False
    text = str(name).lower()
    if _contains_any(text, STRONG_COMPOUND): return True
    if _contains_any(text, ISOLATION): return False
    return _contains_any(text, WEAK_COMPOUND)

primary_data["is_compound"] = [infer_compound(n, c) for n, c in
                               zip(primary_data["name"], primary_data["exercise_category"])]

# check against the dataset's own label (resistance exercises that have one)
check = primary_data[(primary_data["exercise_category"] == "resistance") & primary_data["mechanic"].notna()]
agree = (check["is_compound"] == (check["mechanic"] == "compound")).mean()
print(f"Keyword rule agrees with the dataset's mechanic column for {agree:.1%} of {len(check)} exercises")

## Step 13 - Label one-sided or two-sided
Names with words like "lunge", "single" or "one-arm" are **unilateral** (one limb at a time);
everything else is **bilateral**. Only resistance exercises can be unilateral.

In [ ]:
UNILATERAL = ["lunge", "single", "one-arm", "one arm", "one-leg", "one leg", "unilateral",
              "concentration", "split squat", "step-up", "step up", "bulgarian", "alternating",
              "alternate", "pistol", "kickback"]

def infer_movement_type(name, category):
    if category == "resistance" and _contains_any(str(name).lower(), UNILATERAL):
        return "unilateral"
    return "bilateral"

primary_data["movement_type"] = [infer_movement_type(n, c) for n, c in
                                 zip(primary_data["name"], primary_data["exercise_category"])]
print(primary_data["movement_type"].value_counts())

## Step 14 - Label the joint movement
A keyword table gives each resistance exercise a movement pattern ("row" means
horizontal_pull, "squat" means squat_pattern ...). The first match wins, so specific
words come first. The app uses this to find similar exercises when a user reports
joint pain. Warm-ups, stretches and cardio get "other".

In [ ]:
JOINT_ACTIONS = [
    ("leg curl", "knee_flexion"), ("hamstring curl", "knee_flexion"),
    ("leg extension", "knee_extension"), ("calf", "ankle_plantarflexion"),
    ("deadlift", "hip_hinge"), ("good morning", "hip_hinge"), ("hip thrust", "hip_hinge"),
    ("thrust", "hip_hinge"), ("bridge", "hip_hinge"), ("back extension", "hip_hinge"), ("swing", "hip_hinge"),
    ("squat", "squat_pattern"), ("lunge", "squat_pattern"), ("leg press", "squat_pattern"),
    ("step-up", "squat_pattern"), ("step up", "squat_pattern"),
    ("pulldown", "vertical_pull"), ("pull-down", "vertical_pull"), ("pull-up", "vertical_pull"),
    ("pullup", "vertical_pull"), ("pull up", "vertical_pull"), ("chin-up", "vertical_pull"), ("chin up", "vertical_pull"),
    ("row", "horizontal_pull"), ("face pull", "horizontal_pull"),
    ("overhead press", "vertical_push"), ("shoulder press", "vertical_push"),
    ("military press", "vertical_push"), ("arnold", "vertical_push"),
    ("lateral raise", "shoulder_raise"), ("front raise", "shoulder_raise"),
    ("leg raise", "trunk_flexion"),
    ("bench press", "horizontal_push"), ("chest press", "horizontal_push"), ("push-up", "horizontal_push"),
    ("pushup", "horizontal_push"), ("push up", "horizontal_push"), ("dip", "horizontal_push"),
    ("fly", "chest_fly"), ("flye", "chest_fly"), ("press", "horizontal_push"),
    ("pushdown", "elbow_extension"), ("push-down", "elbow_extension"), ("skull", "elbow_extension"),
    ("tricep", "elbow_extension"), ("curl", "elbow_flexion"),
    ("crunch", "trunk_flexion"), ("sit-up", "trunk_flexion"), ("situp", "trunk_flexion"),
    ("plank", "trunk_stability"), ("twist", "trunk_rotation"),
]

def infer_joint_action(name, category):
    if category != "resistance":
        return "other"
    text = str(name).lower()
    for keyword, action in JOINT_ACTIONS:
        if re.search(r"\b" + re.escape(keyword), text):
            return action
    return "other"

primary_data["joint_action"] = [infer_joint_action(n, c) for n, c in
                                zip(primary_data["name"], primary_data["exercise_category"])]
print(primary_data["joint_action"].value_counts())

## Step 15 - Combine with megaGymDataset (Title + Rating, rating 8 and above)
Only two columns are taken from megaGymDataset: `Title` and `Rating`.
Exercises without a rating, or rated below 8, are dropped. The remaining titles are linked
to the primary data by name: both names are normalised with the same fixed rules, then
compared with a string-similarity score (0-100, word order ignored). A pair counts as the
same exercise when the score is at least 90. Each linked exercise gets the megaGym rating.

`KEEP_ONLY_RATED = True` keeps only the exercises that were linked to a rating of 8+
(the same quality filter the app used on megaGym before).
`KEEP_UNRATED_CATEGORIES` lists categories that are kept even without a rating. It is empty
by default; megaGym rarely rates cardio and stretches, so if the final check in Step 17
warns about missing cardio or stretches, set it to `["cardio", "flexibility"]`.

In [ ]:
# pip install rapidfuzz   (run once if it is not installed)
from rapidfuzz import fuzz, process

MIN_RATING = 8
MATCH_THRESHOLD = 90
KEEP_ONLY_RATED = True
KEEP_UNRATED_CATEGORIES = []          # e.g. ["cardio", "flexibility"]

rated = secondary_data[["Title", "Rating"]].copy()
rated["Rating"] = pd.to_numeric(rated["Rating"], errors="coerce")
print("megaGym rows:", len(rated), "| with a rating:", rated["Rating"].notna().sum())
rated = rated[rated["Rating"] >= MIN_RATING].dropna(subset=["Title"])
rated["Title"] = rated["Title"].apply(lambda v: re.sub(r"\s+", " ", str(v)).strip())
rated["title_key"] = rated["Title"].str.lower()
rated = rated.sort_values("Rating", ascending=False).drop_duplicates("title_key")   # same title twice: keep highest rating
print(f"megaGym exercises rated {MIN_RATING}+:", len(rated))

SYNONYMS = [
    (r"\bdb\b", "dumbbell"), (r"\bbb\b", "barbell"), (r"\bkb\b", "kettlebell"),
    (r"\bsit ?ups?\b", "situp"), (r"\bpush ?ups?\b", "pushup"), (r"\bpull ?ups?\b", "pullup"),
    (r"\bchin ?ups?\b", "chinup"), (r"\bpull ?downs?\b", "pulldown"), (r"\broll ?outs?\b", "rollout"),
    (r"\bflyes?\b|\bflies\b|\bfly\b", "fly"), (r"\btriceps?\b", "triceps"), (r"\bbiceps?\b", "biceps"),
    (r"\bdumbbells\b", "dumbbell"), (r"\bkettlebells\b", "kettlebell"), (r"\blunges\b", "lunge"),
    (r"\bcrunches\b", "crunch"), (r"\braises\b", "raise"), (r"\bcurls\b", "curl"), (r"\brows\b", "row"),
    (r"\bsquats\b", "squat"), (r"\bpresses\b", "press"), (r"\bextensions\b", "extension"),
    (r"\bclimbers\b", "climber"), (r"\bez ?bar\b", "ezbar"), (r"\bsingle ?arm\b|\bone ?arm\b", "onearm"),
    (r"\bsingle ?leg\b|\bone ?leg\b", "oneleg"), (r"\balternating\b|\balternate\b", "alternate"),
]

def match_key(name):
    text = str(name).lower().replace("-", " ").replace("/", " ")
    text = re.sub(r"[^a-z0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    for pattern, replacement in SYNONYMS:
        text = re.sub(pattern, replacement, text)
    return re.sub(r"\s+", " ", text).strip()

primary_keys = primary_data["name"].apply(match_key).tolist()
links = []
for title, rating in zip(rated["Title"], rated["Rating"]):
    best = process.extractOne(match_key(title), primary_keys, scorer=fuzz.token_sort_ratio)
    if best and best[1] >= MATCH_THRESHOLD:
        links.append({"row": best[2], "megagym_title": title, "rating": rating, "match_score": round(best[1], 1)})
links = pd.DataFrame(links).sort_values("rating", ascending=False).drop_duplicates("row")

primary_data["rating"] = np.nan
primary_data["megagym_title"] = ""
primary_data.loc[links["row"], "rating"] = links["rating"].values
primary_data.loc[links["row"], "megagym_title"] = links["megagym_title"].values

print("Primary exercises linked to a megaGym rating of 8+:", primary_data["rating"].notna().sum())
display(primary_data.loc[links["row"], ["name", "megagym_title", "rating"]]
        .assign(score=links["match_score"].values).sort_values("score").head(15))

if KEEP_ONLY_RATED:
    keep = primary_data["rating"].notna() | primary_data["exercise_category"].isin(KEEP_UNRATED_CATEGORIES)
    primary_data = primary_data[keep].reset_index(drop=True)
print("Rows kept:", len(primary_data))
print(primary_data["exercise_category"].value_counts())

## Step 16 - Add hand-written exercises
25 dynamic warm-up moves (category `mobility`, from Muscle & Strength) and 11 static
cool-down stretches (category `flexibility`). Each one is only added if the dataset does
not already have an exercise with the same name.

In [ ]:
anywhere = ["home", "outdoor", "gym"]
HAND_WRITTEN = [
    # --- dynamic warm-up (mobility) - from Muscle & Strength ---
    ("Fast-Paced Walk", "mobility", "full body", "bodyweight", anywhere, "beginner",
     "Walk at a brisk pace to elevate your core body temperature and blood flow."),
    ("Plantar Fascia w/ Lacrosse Ball", "mobility", "calves", "lacrosse ball", anywhere, "beginner",
     "While standing or seated, roll a lacrosse ball under the arch of each foot to release tension."),
    ("Donkey Calf Raise", "mobility", "calves", "bodyweight", anywhere, "beginner",
     "Hinge at the hips (placing hands on a bench or wall) and perform controlled calf raises."),
    ("Dumbbell Goblet Squat", "mobility", "quads", "dumbbells", ["home", "gym"], "beginner",
     "Hold a dumbbell vertically against your chest and perform a controlled squat to open the hips."),
    ("Bodyweight Walking Lunge", "mobility", "quads", "bodyweight", anywhere, "beginner",
     "Step forward into a lunge, alternating legs as you walk forward."),
    ("Side Plank", "mobility", "core", "bodyweight", anywhere, "beginner",
     "Prop yourself up on one forearm and the side of your foot, keeping your body perfectly straight."),
    ("Plyometric Push Up", "mobility", "chest", "bodyweight", anywhere, "intermediate",
     "Perform a push-up with enough explosive force to briefly lift your hands off the ground."),
    ("Rack Lat Stretch", "mobility", "back", "machine", ["gym"], "beginner",
     "Hold onto a squat rack or sturdy pole, hinge at the hips, and pull back gently to stretch your lats."),
    ("Bodyweight Standing Calf Raise", "mobility", "calves", "bodyweight", anywhere, "beginner",
     "Stand tall and push up onto your toes, then lower with control to warm up the ankles."),
    ("IT Band Foam Roller", "mobility", "quads", "foam roll", ["home", "gym"], "beginner",
     "Lie on your side and slowly roll the foam roller along your outer thigh."),
    ("Bodyweight Single-Leg Deadlift", "mobility", "hamstrings", "bodyweight", anywhere, "beginner",
     "Hinge at the hips on one leg, keeping your back straight and the other leg extended behind you."),
    ("Prisoner Squat", "mobility", "quads", "bodyweight", anywhere, "beginner",
     "Place your hands behind your head and perform a bodyweight squat."),
    ("Prisoner Squat (Hold at the Bottom)", "mobility", "quads", "bodyweight", anywhere, "beginner",
     "Perform a prisoner squat and hold the bottom position isometrically to improve hip mobility."),
    ("Superman", "mobility", "back", "bodyweight", anywhere, "beginner",
     "Lie face down and lift your arms and legs off the floor simultaneously, squeezing the lower back."),
    ("Plank", "mobility", "core", "bodyweight", anywhere, "beginner",
     "Hold a push-up position on your forearms with a straight back and tight core."),
    ("Band Pull-Apart", "mobility", "shoulders", "resistance band", anywhere, "beginner",
     "Hold a resistance band with straight arms and pull it apart until it touches your chest."),
    ("Warm-up Push Up", "mobility", "chest", "bodyweight", anywhere, "beginner",
     "Lower your body until your chest is just above the floor, then push back up."),
    ("Warm-up Bench Dips", "mobility", "triceps", "bench", ["gym", "home"], "beginner",
     "Place your hands on a bench behind you, lower your hips toward the floor, then press back up."),
    ("Arm Circles (Clockwise)", "mobility", "shoulders", "bodyweight", anywhere, "beginner",
     "Extend your arms straight out to the sides and make controlled circles clockwise."),
    ("Arm Circles (Counter Clockwise)", "mobility", "shoulders", "bodyweight", anywhere, "beginner",
     "Extend your arms straight out to the sides and make controlled circles counter-clockwise."),
    ("Eccentric Only Pull Up", "mobility", "back", "bodyweight", ["gym", "outdoor"], "intermediate",
     "Jump to the top of a pull-up position, then lower yourself down as slowly as possible."),
    ("Seated Zottman Curl", "mobility", "biceps", "dumbbells", ["gym", "home"], "beginner",
     "Curl dumbbells up with palms facing up, rotate wrists at the top, and lower with palms facing down."),
    ("90/90 Hip Crossover", "mobility", "glutes", "bodyweight", anywhere, "beginner",
     "Sit with both legs bent at 90 degrees and rotate your hips/knees from side to side on the floor."),
    ("Warm-up Sit Up", "mobility", "core", "bodyweight", anywhere, "beginner",
     "Lie on your back, knees bent, and sit all the way up to warm up the abdominals."),
    ("Jump Squat", "mobility", "quads", "bodyweight", anywhere, "beginner",
     "Perform a bodyweight squat and explode upward into a jump to activate the nervous system."),
    # --- static cool-down stretches (flexibility) ---
    ("Cross-Body Shoulder Stretch", "flexibility", "shoulders", "bodyweight", anywhere, "beginner",
     "1. Bring one arm across your chest.\n2. Hold it with the other arm.\n3. Hold the stretch, then switch."),
    ("Child's Pose", "flexibility", "back", "bodyweight", anywhere, "beginner",
     "1. Kneel and sit back on your heels.\n2. Reach your arms forward along the floor.\n3. Relax and breathe deeply."),
    ("Overhead Side Reach Lat Stretch", "flexibility", "back", "bodyweight", anywhere, "beginner",
     "1. Raise one arm overhead.\n2. Lean gently to the opposite side.\n3. Hold, then switch."),
    ("Standing Quad Stretch", "flexibility", "quads", "bodyweight", anywhere, "beginner",
     "1. Stand on one leg (hold a wall).\n2. Pull the other heel toward your glute.\n3. Keep knees together and hold, then switch."),
    ("Seated Hamstring Stretch", "flexibility", "hamstrings", "bodyweight", anywhere, "beginner",
     "1. Sit with one leg straight, the other bent.\n2. Hinge forward from your hips.\n3. Hold, then switch."),
    ("Figure-Four Glute Stretch", "flexibility", "glutes", "bodyweight", anywhere, "beginner",
     "1. Lie on your back, cross one ankle over the opposite knee.\n2. Pull the thigh toward your chest.\n3. Hold, then switch."),
    ("Cobra Stretch", "flexibility", "core", "bodyweight", anywhere, "beginner",
     "1. Lie face down, hands under shoulders.\n2. Gently press up, keeping hips on the floor.\n3. Hold and breathe."),
    ("Doorway Biceps Stretch", "flexibility", "biceps", "bodyweight", anywhere, "beginner",
     "1. Place your palm on a wall, arm straight, thumb down.\n2. Slowly turn your body away.\n3. Hold, then switch."),
    ("Overhead Triceps Stretch", "flexibility", "triceps", "bodyweight", anywhere, "beginner",
     "1. Raise one arm and bend the elbow behind your head.\n2. Gently press the elbow with your other hand.\n3. Hold, then switch."),
    ("Wall Calf Stretch", "flexibility", "calves", "bodyweight", anywhere, "beginner",
     "1. Stand facing a wall, one foot back with the heel down.\n2. Lean forward until you feel the calf stretch.\n3. Hold, then switch."),
    ("Standing Forward Fold", "flexibility", "full body", "bodyweight", anywhere, "beginner",
     "1. Stand with feet hip-width apart.\n2. Fold forward from the hips, knees slightly bent.\n3. Let your head and arms hang and breathe."),
]

existing = set(primary_data["name"].apply(name_key))
added, skipped = [], []
for name, category, muscle, equipment, locations, level, text in HAND_WRITTEN:
    if name_key(name) in existing:
        skipped.append(name)
        continue
    added.append({
        "name": name, "exercise_category": category, "target_muscle": muscle.title(),
        "major_muscle_group": major_muscle_group(muscle), "equipment_needed": equipment,
        "location_suitability": locations, "difficulty_level": level, "instructions": text,
        "is_compound": infer_compound(name, category), "movement_type": infer_movement_type(name, category),
        "joint_action": infer_joint_action(name, category), "image_1": "", "image_2": "",
        "source": "hand-written",
    })

primary_data["source"] = "Free Exercise DB"
final_kb = pd.concat([primary_data, pd.DataFrame(added)], ignore_index=True)
print("Hand-written exercises added:", len(added), "| skipped (name already exists):", skipped)
print("Total exercises:", len(final_kb))

## Step 16b - Bring back no-equipment cardio (optional, on by default)
Home users need cardio that needs no machine. In the Free Exercise DB, "Rope Jumping" was removed
in Step 7 because its equipment is "other", and "Trail Running/Walking" has no megaGym rating.
This step re-adds both from the original file, with their real instructions and photos,
and relabels Rope Jumping's equipment as "jump rope" (an option in the app's profile form).
Set `ADD_HOME_CARDIO = False` to skip it.

In [ ]:
ADD_HOME_CARDIO = True
HOME_CARDIO = {"Rope Jumping": "jump rope", "Trail Running/Walking": "bodyweight"}   # name -> equipment

if ADD_HOME_CARDIO:
    raw = pd.read_csv(r"C:\Users\RICHARD\free_exercise_db.csv")
    extra = raw[raw["name"].isin(HOME_CARDIO)].copy()
    extra = extra[~extra["name"].apply(name_key).isin(final_kb["name"].apply(name_key))]
    extra["instructions"] = extra["instructions"].apply(clean_text)
    extra["exercise_category"] = "cardio"
    extra["difficulty_level"] = extra["level"].apply(normalise_level)
    extra["equipment_needed"] = extra["name"].map(HOME_CARDIO)
    extra["location_suitability"] = [ANYWHERE] * len(extra)
    extra["target_muscle"] = "Full Body"
    extra["major_muscle_group"] = "other"
    extra["is_compound"], extra["movement_type"], extra["joint_action"] = False, "bilateral", "other"
    extra["source"] = "Free Exercise DB"
    final_kb = pd.concat([final_kb, extra], ignore_index=True)
    print("Added:", extra["name"].tolist(), "| total exercises:", len(final_kb))

## Step 16c - Photos for the hand-written exercises
The hand-written warm-ups and cool-downs have no photos. Each one is linked to the Free Exercise DB
entry that shows the **same movement**, using the table below. Every pair was checked by looking at
the photos; a pair was only used when the photos show the same movement (a different variation,
equipment or body position was rejected). The exercise keeps its own name and instructions; only
the two photos (start and end position) are taken from the linked entry.

`REQUIRE_PHOTOS = True` then removes every exercise that still has no two photos, so every exercise
in the app shows two pictures. Set it to `False` to keep them without photos.

In [ ]:
REQUIRE_PHOTOS = True

# hand-written exercise  ->  Free Exercise DB entry whose photos show the same movement
PHOTO_SOURCE = {
    # warm-ups
    "Fast-Paced Walk": "Trail Running/Walking",
    "Plantar Fascia w/ Lacrosse Ball": "Foot-SMR",
    "Dumbbell Goblet Squat": "Goblet Squat",                  # photo shows a kettlebell, same movement
    "Bodyweight Walking Lunge": "Bodyweight Walking Lunge",
    "Side Plank": "Side Bridge",
    "Plyometric Push Up": "Plyo Push-up",
    "IT Band Foam Roller": "Iliotibial Tract-SMR",
    "Prisoner Squat": "Bodyweight Squat",                     # hands behind the head in the photo
    "Prisoner Squat (Hold at the Bottom)": "Bodyweight Squat",
    "Superman": "Superman",
    "Plank": "Plank",
    "Band Pull-Apart": "Band Pull Apart",
    "Warm-up Push Up": "Pushups",
    "Warm-up Bench Dips": "Bench Dips",
    "Arm Circles (Clockwise)": "Arm Circles",
    "Arm Circles (Counter Clockwise)": "Arm Circles",
    "Eccentric Only Pull Up": "Pullups",
    "Seated Zottman Curl": "Zottman Curl",                    # photo is standing, same movement
    "Warm-up Sit Up": "Sit-Up",
    "Jump Squat": "Freehand Jump Squat",
    # cool-down stretches
    "Cross-Body Shoulder Stretch": "Shoulder Stretch",
    "Overhead Side Reach Lat Stretch": "Standing Lateral Stretch",
    "Seated Hamstring Stretch": "Seated Floor Hamstring Stretch",
    "Figure-Four Glute Stretch": "Ankle On The Knee",
    "Overhead Triceps Stretch": "Triceps Stretch",
    "Wall Calf Stretch": "Calf Stretch Hands Against Wall",
    "Standing Forward Fold": "Standing Toe Touches",
}
# Not linked (no Free Exercise DB entry shows the same movement):
# Donkey Calf Raise and Bodyweight Standing Calf Raise (source photos use a machine),
# Bodyweight Single-Leg Deadlift, Rack Lat Stretch, 90/90 Hip Crossover,
# Standing Quad Stretch, Cobra Stretch, Doorway Biceps Stretch.

raw_photos = pd.read_csv(r"C:\Users\RICHARD\free_exercise_db.csv").set_index("name")[["image_1", "image_2"]]
missing = [v for v in PHOTO_SOURCE.values() if v not in raw_photos.index]
assert not missing, f"Not found in Free Exercise DB: {missing}"

final_kb["photo_source"] = np.where(final_kb["source"] == "Free Exercise DB", final_kb["name"], "")
for exercise, source_name in PHOTO_SOURCE.items():
    rows = final_kb["name"] == exercise
    if rows.any():
        final_kb.loc[rows, ["image_1", "image_2"]] = raw_photos.loc[source_name, ["image_1", "image_2"]].values
        final_kb.loc[rows, "photo_source"] = source_name

final_kb[["image_1", "image_2"]] = final_kb[["image_1", "image_2"]].fillna("")
no_photos = final_kb[(final_kb["image_1"] == "") | (final_kb["image_2"] == "")]
print("Hand-written exercises given photos:", int(final_kb["name"].isin(PHOTO_SOURCE).sum()))
print("Exercises still without two photos:", no_photos["name"].tolist())
if REQUIRE_PHOTOS:
    final_kb = final_kb[(final_kb["image_1"] != "") & (final_kb["image_2"] != "")].reset_index(drop=True)
    print("Removed them. Total exercises:", len(final_kb))

## Step 17 - Final check and save
Keep only the columns the app needs, check that nothing important is missing, show how many
exercises there are per category and location, and save `exercise_kb_final.csv`.
That file goes into the GitHub repository next to the app.

In [ ]:
final_kb = final_kb.rename(columns={"name": "exercise_name"})
APP_COLUMNS = ["exercise_name", "exercise_category", "target_muscle", "major_muscle_group",
               "equipment_needed", "location_suitability", "difficulty_level", "instructions",
               "is_compound", "movement_type", "joint_action", "image_1", "image_2",
               "rating", "megagym_title", "source", "photo_source"]
final_kb = final_kb[APP_COLUMNS].copy()
final_kb[["image_1", "image_2", "megagym_title", "photo_source"]] = final_kb[["image_1", "image_2", "megagym_title", "photo_source"]].fillna("")

# checks
assert final_kb["exercise_name"].is_unique
assert final_kb["instructions"].fillna("").str.strip().ne("").all()
assert final_kb["exercise_category"].isin(["resistance", "cardio", "plyometrics", "mobility", "flexibility"]).all()
assert final_kb["difficulty_level"].isin(["beginner", "intermediate", "advanced"]).all()
if REQUIRE_PHOTOS:
    assert ((final_kb["image_1"] != "") & (final_kb["image_2"] != "")).all(), "some exercises have no photos"

# coverage: options available to a home user with no equipment, per category
home_bw = final_kb[final_kb["equipment_needed"].eq("bodyweight")
                   & final_kb["location_suitability"].apply(lambda p: "home" in p)]
summary = pd.DataFrame({
    "all exercises": final_kb["exercise_category"].value_counts(),
    "home, no equipment": home_bw["exercise_category"].value_counts(),
}).fillna(0).astype(int)
display(summary)
MIN_HOME_OPTIONS = {"cardio": 1, "mobility": 3, "flexibility": 3}
for category, minimum in MIN_HOME_OPTIONS.items():
    n = summary["home, no equipment"].get(category, 0)
    if n < minimum:
        print(f"WARNING: only {n} no-equipment '{category}' option(s) for home users.")

out = final_kb.copy()
out["location_suitability"] = out["location_suitability"].apply(json.dumps)   # store the list as text
out.to_csv("exercise_kb_final.csv", index=False, encoding="utf-8")
print("Saved exercise_kb_final.csv with", len(out), "exercises")

## Step 18 - Overview charts (for the report)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=final_kb, y="exercise_category", order=final_kb["exercise_category"].value_counts().index, ax=axes[0])
axes[0].set_title("Exercises per category")
sns.countplot(data=final_kb[final_kb["exercise_category"] == "resistance"], y="major_muscle_group",
              order=final_kb.loc[final_kb["exercise_category"] == "resistance", "major_muscle_group"].value_counts().index, ax=axes[1])
axes[1].set_title("Resistance exercises per muscle group")
plt.tight_layout()
plt.show()